# ML Assignment 2 — Breast Cancer Classification
**M.Tech (AIML/DSE) — Machine Learning — BITS Pilani WILP**

This notebook:
1. Loads the **Breast Cancer Wisconsin (Diagnostic)** dataset (569 instances, 30 features, binary classification)
2. Trains 5 classifiers: Logistic Regression, Decision Tree, kNN, Naive Bayes, Random Forest (ensemble)
3. Computes Accuracy, AUC, Precision, Recall, F1, and MCC for each
4. Saves the trained models, scaler, test split, and metrics table — ready to download and drop into your Streamlit app repo

> Run all cells top to bottom (**Runtime → Run all**). At the end, a `project-folder.zip` is created and downloaded automatically with everything you need for GitHub.


## 1. Install & import dependencies

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib


In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Load the dataset

**Dataset:** Breast Cancer Wisconsin (Diagnostic) — built into `scikit-learn`, originally from the UCI ML Repository.
- Instances: 569 (≥ 500 required ✅)
- Features: 30 (≥ 12 required ✅)
- Target: binary — `0 = malignant`, `1 = benign`


In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame
feature_names = list(data.feature_names)

print("Shape:", df.shape)
print("Class balance:\n", df['target'].value_counts())
df.head()


## 3. Train / test split

The test split is exported as `test_data.csv` — this is the file you upload into the Streamlit app.

In [ ]:
X = df[feature_names]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

os.makedirs('project-folder/model', exist_ok=True)

test_export = X_test.copy()
test_export['target'] = y_test.values
test_export.to_csv('project-folder/test_data.csv', index=False)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
test_export.head()


## 4. Scale features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'project-folder/model/scaler.pkl')
print("Scaler saved.")


## 5. Train all 5 models and evaluate

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "kNN": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    predictions[name] = (y_pred, y_prob)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })

    fname = f"project-folder/model/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, fname)
    print(f"Trained + saved: {name} -> {fname}")

results_df = pd.DataFrame(results).round(4)
results_df.to_csv('project-folder/model/metrics_summary.csv', index=False)
results_df


## 6. Comparison chart

In [ ]:
metrics_to_plot = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
ax = results_df.set_index("Model")[metrics_to_plot].plot(
    kind="bar", figsize=(11, 5), rot=20
)
ax.set_ylim(0, 1.05)
ax.set_title("Model Comparison Across Metrics")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 7. Confusion matrices for each model

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, (name, (y_pred, _)) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["Malig(0)", "Benign(1)"],
                yticklabels=["Malig(0)", "Benign(1)"])
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 8. Full classification reports

In [ ]:
for name, (y_pred, _) in predictions.items():
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["Malignant", "Benign"]))
    print()


## 9. Write `app.py`, `requirements.txt`, and `README.md`

This regenerates the rest of the project folder (Streamlit app + docs) right here in Colab, so everything
needed for your GitHub repo ends up in one place: `project-folder/`.


In [ ]:

app_py = r"""
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report,
)

st.set_page_config(page_title="Breast Cancer Classifier Demo", layout="wide")

MODEL_DIR = "model"
MODEL_FILES = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "kNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "Random Forest (Ensemble)": "random_forest.pkl",
}
TARGET_COL = "target"

@st.cache_resource
def load_model(model_key):
    return joblib.load(os.path.join(MODEL_DIR, MODEL_FILES[model_key]))

@st.cache_resource
def load_scaler():
    return joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))

st.title("Breast Cancer Classification - Model Demo")
st.caption("Assignment 2 - Machine Learning - Dataset: Breast Cancer Wisconsin (Diagnostic)")

st.sidebar.header("1. Upload Test Data")
uploaded_file = st.sidebar.file_uploader("Upload test_data.csv", type=["csv"])

st.sidebar.header("2. Choose Model")
model_choice = st.sidebar.selectbox("Select a classification model", list(MODEL_FILES.keys()))

if uploaded_file is None:
    st.info("Upload a test CSV file from the sidebar to get started.")
    st.stop()

data = pd.read_csv(uploaded_file)
st.subheader("Preview of Uploaded Data")
st.dataframe(data.head(), use_container_width=True)

has_labels = TARGET_COL in data.columns
feature_cols = [c for c in data.columns if c != TARGET_COL]

model = load_model(model_choice)
scaler = load_scaler()

X = data[feature_cols]
X_scaled = scaler.transform(X)

y_pred = model.predict(X_scaled)
y_prob = model.predict_proba(X_scaled)[:, 1]

st.subheader(f"Predictions - {model_choice}")
pred_display = data.copy()
pred_display["Predicted"] = y_pred
pred_display["Probability (class=1)"] = y_prob.round(4)
st.dataframe(pred_display.head(20), use_container_width=True)

st.subheader("Evaluation Metrics")
if has_labels:
    y_true = data[TARGET_COL]
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    cols = st.columns(6)
    for c, n, v in zip(cols, ["Accuracy","AUC","Precision","Recall","F1","MCC"], [acc,auc,prec,rec,f1,mcc]):
        c.metric(n, f"{v:.4f}")

    st.subheader("Confusion Matrix")
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Malignant (0)", "Benign (1)"],
                yticklabels=["Malignant (0)", "Benign (1)"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    st.pyplot(fig)

    st.subheader("Classification Report")
    report = classification_report(y_true, y_pred, output_dict=True)
    st.dataframe(pd.DataFrame(report).transpose().round(4), use_container_width=True)
else:
    st.warning("No 'target' column found - only predictions are shown above.")

st.divider()
st.caption("Built for Assignment 2 - Machine Learning, M.Tech (AIML/DSE), BITS Pilani WILP")
"""

with open("project-folder/app.py", "w") as f:
    f.write(app_py)

requirements_txt = "streamlit\\nscikit-learn\\nnumpy\\npandas\\nmatplotlib\\nseaborn\\njoblib\\n"
with open("project-folder/requirements.txt", "w") as f:
    f.write(requirements_txt)

print("app.py and requirements.txt written to project-folder/")


In [ ]:

readme_md = f"""# Breast Cancer Classification — ML Assignment 2

## a. Problem Statement
This project implements and compares five classification models (Logistic
Regression, Decision Tree, k-Nearest Neighbors, Naive Bayes, and Random
Forest as an ensemble model) to predict whether a breast tumor is malignant
or benign, and exposes them through an interactive Streamlit web app.

## b. Dataset Description
- Name: Breast Cancer Wisconsin (Diagnostic) Data Set
- Source: UCI Machine Learning Repository / scikit-learn `load_breast_cancer`
- Instances: 569 (>= 500 required)
- Features: 30 numeric features (>= 12 required)
- Target: Binary — 0 = malignant, 1 = benign
- Train/test split: 80/20, stratified, random_state=42
- `test_data.csv` in this repo is the held-out 20% test split

## c. GitHub Repository Link
> TODO: paste your repo URL here after pushing

## d. Models Used

### Comparison Table
{results_df.to_markdown(index=False)}

### Observations
| ML Model Name | Observation about model performance |
|---|---|
| Logistic Regression | Best overall — classes are close to linearly separable once scaled. |
| Decision Tree | Weakest — a single unpruned tree overfits and generalizes worst. |
| kNN | Very strong, high recall — distance-based split works well on compact clusters. |
| Naive Bayes | Middling — independence assumption is violated by correlated features. |
| Random Forest (Ensemble) | Solid and well-balanced, reduces the overfitting seen in the single tree. |
| Overall Winner | **Logistic Regression** based on this run's metrics. |

*(Re-run this notebook — results may shift slightly depending on random seed / sklearn version; update this table with your own numbers before submitting.)*

## Streamlit App Features
- Dataset upload (CSV)
- Model selection dropdown
- Live evaluation metrics
- Confusion matrix + classification report

## How to Run Locally
```
pip install -r requirements.txt
streamlit run app.py
```

## Live Streamlit App Link
> TODO: paste your deployed app URL here
"""

with open("project-folder/README.md", "w") as f:
    f.write(readme_md)

print("README.md written to project-folder/")


## 10. Zip everything and download

This creates `project-folder.zip` containing `app.py`, `requirements.txt`, `README.md`, `test_data.csv`, and the `model/` folder — ready to push to GitHub.

In [ ]:
import shutil
shutil.make_archive('project-folder', 'zip', '.', 'project-folder')

from google.colab import files
files.download('project-folder.zip')


## Next steps
1. Unzip `project-folder.zip` on your machine (or directly on the BITS Virtual Lab, if required for the screenshot step).
2. `git init`, `git add .`, `git commit`, then push to a new GitHub repository.
3. Deploy `app.py` on [Streamlit Community Cloud](https://streamlit.io/cloud).
4. Update the `TODO` lines in `README.md` with your real GitHub and Streamlit links, commit again.
5. Assemble your final submission PDF: GitHub link → Streamlit link → BITS Lab screenshot → README content.
